<a href="https://colab.research.google.com/github/SaraBatistaPereira/GSI073/blob/main/aula4modificadoomodelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Instalar as ferramentas necessárias
!pip -q install transformers datasets accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "distilbert/distilgpt2"

# 2. Preparar os dados (Exatamente como na sua aula)
docs = [
    "UFU forma estudantes de IA aplicada para muitas áreas.",
    "Um transformer decoder-only prevê o próximo token com auto-regressão.",
    "Fine-tuning ajusta um modelo pré-treinado para um domínio específico.",
    "Temperatura controla aleatoriedade; top-k e top-p controlam o corte do vocabulário.",
    "A Universidade Federal de Uberlândia é top demais."
]
ds = Dataset.from_dict({"text": docs})
tok = AutoTokenizer.from_pretrained(model_id)
tok.pad_token = tok.eos_token

# --- TESTE DO MODELO BASE (ANTES DO TREINO) ---
model_base = AutoModelForCausalLM.from_pretrained(model_id).to(device)
prompt = "O processo de fine-tuning serve para"
inputs = tok(prompt, return_tensors="pt").to(device)

print("\n=== RESPOSTA DO MODELO BASE (ANTES) ===")
with torch.no_grad():
    out_base = model_base.generate(**inputs, max_new_tokens=20)
print(tok.decode(out_base[0], skip_special_tokens=True))

# 3. Treinar o modelo (Rápido)
def tokenize_fn(batch): return tok(batch["text"], truncation=True, max_length=64)
tokenized_ds = ds.map(tokenize_fn, batched=True)
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

args = TrainingArguments(output_dir="./result", num_train_epochs=20, per_device_train_batch_size=2, report_to="none")
trainer = Trainer(model=model_base, args=args, train_dataset=tokenized_ds, data_collator=collator)
trainer.train()

# --- TESTE DO MODELO AJUSTADO (DEPOIS) ---
print("\n=== RESPOSTA DO MODELO AJUSTADO (DEPOIS) ===")
with torch.no_grad():
    out_tuned = model_base.generate(**inputs, max_new_tokens=20)
print(tok.decode(out_tuned[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



=== RESPOSTA DO MODELO AJUSTADO (DEPOIS) ===
O processo de fine-tuning serve para o pré-treinado para o pré-treinado para o pré-
